In [1]:
import torch
import torch.nn.functional as F



In [2]:
def scaled_dot_product_attention(Q, K, V):
    scores = Q @ K.transpose(-2, -1)   # 왜 transpose? 모양 직접 따져봐
    d_k = Q.size(-1)           # 마지막 차원이 d_k
    scores = scores / (d_k ** 0.5)      
    weights = F.softmax(scores, dim=-1)   # dim=-1이 왜 맞을까?
    output = weights @ V       # 모양 확인

    return output, weights   # weights도 반환하면 attention 시각화에 씀

In [ ]:
torch.manual_seed(0)
Q = torch.randn(2, 4)   # 질의 2개, d_k=4
K = torch.randn(3, 4)   # 키 3개
V = torch.randn(3, 5)   # 값 3개, d_v=5
out, w = scaled_dot_product_attention(Q, K, V)
print(out.shape)   # 예상: (2, 5) — 직접 예측하고 맞춰봐
print(w.shape)     # 예상: (2, 3)
print(w.sum(dim=-1))  # 각 행 합이 1인지 확인 (softmax 검증)

torch.Size([2, 5])
torch.Size([2, 3])
tensor([1., 1.])


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0   # 쪼개지려면 나누어떨어져야
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads   # head당 차원

        # Q, K, V를 만드는 선형변환 + 출력 선형변환
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        batch, seq_len, _ = x.size()

        # Step 1: 선형변환으로 Q, K, V 만들기
        Q = self.W_q(x)   # (batch, seq_len, d_model)
        K = self.W_k(x)
        V = self.W_v(x)

        # Step 2: h개 head로 쪼개기 ← 여기가 핵심, 직접 채워
        # (batch, seq_len, d_model) → (batch, num_heads, seq_len, d_k)
        # 힌트: .view(batch, seq_len, num_heads, d_k) 후 .transpose(1, 2)
        Q = Q.view(batch, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # Step 3: 각 head에 scaled dot-product attention
        # A에서 짠 거랑 같음! scores = Q @ K^T / √d_k → softmax → @ V
        # 이제 Q,K,V가 (batch, num_heads, seq_len, d_k)라 마지막 두 축이 핵심
        scores = Q@K.transpose(-2,-1)/(self.d_k**0.5)
        weights = F.softmax(scores, dim=-1)
        attn = weights@V   # weights @ V → (batch, num_heads, seq_len, d_k)

        # Step 4: head 합치기 (concat) ← Step 2의 역연산
        # (batch, num_heads, seq_len, d_k) → (batch, seq_len, d_model)
        # 힌트: .transpose(1, 2) 후 .contiguous().view(batch, seq_len, d_model)
        out = attn.transpose(1, 2).contiguous().view(batch, seq_len, self.d_model)

        # Step 5: 출력 선형변환
        return self.W_o(out)

In [5]:
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=8, num_heads=2)
x = torch.randn(1, 5, 8)   # batch=1, seq_len=5, d_model=8
out = mha(x)
print(out.shape)   # 예상? 직접 맞춰봐

torch.Size([1, 5, 8])
